<a href="https://colab.research.google.com/github/askSayyam/CrisisLens/blob/main/Crisislens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
DRIVE = '/content/drive/MyDrive/CrisisLens/'
print("Drive connected. Files in your folder:")

import os
for f in os.listdir(DRIVE):
    size = os.path.getsize(DRIVE + f) / (1024*1024)
    print(f"  {f:<40} {size:.1f} MB")

Drive connected. Files in your folder:
  disastir_corpus.csv                      206.8 MB
  disastir_queries.csv                     1.8 MB
  disastir_qrels.csv                       0.0 MB
  semeval2025_task7_data.zip               94.4 MB


In [11]:
import pandas as pd

corpus  = pd.read_csv(DRIVE + 'disastir_corpus.csv')
queries = pd.read_csv(DRIVE + 'disastir_queries.csv')
qrels   = pd.read_csv(DRIVE + 'disastir_qrels.csv')

print("Corpus shape: ",  corpus.shape)
print("Queries shape:", queries.shape)
print("Qrels shape:  ",  qrels.shape)
print("\nCorpus columns: ",  corpus.columns.tolist())
print("Queries columns:", queries.columns.tolist())
print("Qrels columns:  ",  qrels.columns.tolist())

Corpus shape:  (239704, 2)
Queries shape: (9600, 2)
Qrels shape:   (2757377, 3)

Corpus columns:  ['corpus_id', 'text']
Queries columns: ['query_id', 'user_query']
Qrels columns:   ['user_query', 'passage', 'score']


In [12]:
import ast

# Load SemEval files
posts_df = pd.read_csv(DRIVE + 'semeval/train_dev_sets/posts.csv')
fc_df    = pd.read_csv(DRIVE + 'semeval/train_dev_sets/fact_checks.csv')
pairs_df = pd.read_csv(DRIVE + 'semeval/train_dev_sets/pairs_dev_crosslingual.csv')

print("Posts shape:       ", posts_df.shape)
print("Fact-checks shape: ", fc_df.shape)
print("Pairs shape:       ", pairs_df.shape)
print("\nPosts columns:      ", posts_df.columns.tolist())
print("Fact-check columns: ", fc_df.columns.tolist())

Posts shape:        (24431, 5)
Fact-checks shape:  (153743, 4)
Pairs shape:        (651, 2)

Posts columns:       ['post_id', 'instances', 'ocr', 'verdicts', 'text']
Fact-check columns:  ['fact_check_id', 'claim', 'instances', 'title']


In [15]:
import ast

# ── Parser function ──
def parse_tuple(val):
    if pd.isna(val): return None, 'unk'
    try:
        t = ast.literal_eval(str(val))
        text = t[0]
        lang = t[2][0][0] if t[2] else 'unk'
        return text, lang
    except:
        return str(val), 'unk'

# ── Parse posts ──
posts_df[['text_clean', 'lang']] = posts_df['text'].apply(
    lambda x: pd.Series(parse_tuple(x)))

# ── Parse fact-checks ──
fc_df[['claim_clean', 'claim_lang']] = fc_df['claim'].apply(
    lambda x: pd.Series(parse_tuple(x)))
fc_df[['title_clean', 'title_lang']] = fc_df['title'].apply(
    lambda x: pd.Series(parse_tuple(x)))

fc_df['text_clean'] = (fc_df['claim_clean'].fillna('') + ' ' +
                        fc_df['title_clean'].fillna('')).str.strip()
fc_df['lang'] = fc_df['claim_lang']

# ── Verify (fixed — skip null rows) ──
first_valid_post = posts_df[posts_df['text_clean'].notna()].iloc[0]
first_valid_fc   = fc_df[fc_df['text_clean'].notna()].iloc[0]

print("Sample parsed post:")
print(f"  lang: {first_valid_post['lang']}")
print(f"  text: {first_valid_post['text_clean'][:100]}")

print(f"\nSample parsed fact-check:")
print(f"  lang: {first_valid_fc['lang']}")
print(f"  text: {first_valid_fc['text_clean'][:100]}")

print(f"\nPosts — total: {len(posts_df)} | valid text: {posts_df['text_clean'].notna().sum()}")
print(f"Fact-checks — total: {len(fc_df)} | valid text: {fc_df['text_clean'].notna().sum()}")

print(f"\nLanguage distribution in posts:")
print(posts_df['lang'].value_counts().head(10))



Sample parsed post:
  lang: spa
  text: "Don Quijote soy, y mi profesión la de andante caballería. Son mis leyes, el deshacer entuertos, pro

Sample parsed fact-check:
  lang: eng
  text: Are avocados good for you?

Posts — total: 24431 | valid text: 21977
Fact-checks — total: 153743 | valid text: 153743

Language distribution in posts:
lang
spa    6195
eng    4694
por    2869
unk    2454
fra    1911
msa    1328
ara     891
hin     748
deu     692
tha     607
Name: count, dtype: int64


In [17]:
# ── DisastIR part ──
disastir_part = pd.DataFrame({
    'uid'     : 'dis_' + corpus['corpus_id'].astype(str),
    'text'    : corpus['text'],
    'lang'    : 'eng',
    'source'  : 'disastir',
    'category': 'disaster'
})

# ── SemEval fact-checks part ──
semeval_part = pd.DataFrame({
    'uid'     : 'fc_' + fc_df['fact_check_id'].astype(str),
    'text'    : fc_df['text_clean'],
    'lang'    : fc_df['lang'],
    'source'  : 'semeval',
    'category': 'fact_check'
})

# ── Merge both ──
unified = pd.concat([disastir_part, semeval_part], ignore_index=True)

# ── Drop nulls and empty rows ──
unified = unified.dropna(subset=['text'])
unified = unified[unified['text'].str.strip().str.len() > 10]
unified = unified.reset_index(drop=True)

print(f"DisastIR passages : {len(disastir_part):>7,}")
print(f"SemEval fact-checks: {len(semeval_part):>7,}")
print(f"Total unified     : {len(unified):>7,}")
print(f"\nSample rows:")
print(unified.head(3).to_string())
print(unified.tail(3).to_string())

DisastIR passages : 239,704
SemEval fact-checks: 153,743
Total unified     : 393,429

Sample rows:
            uid                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [18]:
unified.to_csv(DRIVE + 'unified_corpus.csv', index=False)
print(f"Saved unified_corpus.csv to Drive ✓")
print(f"Total rows saved: {len(unified):,}")

Saved unified_corpus.csv to Drive ✓
Total rows saved: 393,429


In [20]:
import re

# ── Preprocessor functions ──
def deep_clean(text):
    """For unified corpus — full cleaning"""
    if not isinstance(text, str): return ''
    text = text.strip()
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'\s+', ' ', text)           # collapse spaces/newlines
    text = re.sub(r'[^\w\s\u0600-\u06FF\u0750-\u077F,.!?]', ' ', text)  # keep Arabic script
    text = text.strip()
    return text

def light_clean(text):
    """For queries and posts — minimal cleaning"""
    if not isinstance(text, str): return ''
    text = text.strip()
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'\s+', ' ', text)           # collapse spaces
    text = text.strip()
    return text

# ── 1. Clean unified corpus ──
print("Cleaning unified corpus...")
unified['text'] = unified['text'].apply(deep_clean)
unified = unified[unified['text'].str.len() > 10].reset_index(drop=True)
unified.to_csv(DRIVE + 'unified_corpus.csv', index=False)
print(f"  ✓ unified_corpus.csv saved — {len(unified):,} rows")

# ── 2. Clean DisastIR queries ──
print("Cleaning DisastIR queries...")
queries['user_query'] = queries['user_query'].apply(light_clean)
queries = queries[queries['user_query'].str.len() > 3].reset_index(drop=True)
queries.to_csv(DRIVE + 'queries.csv', index=False)
print(f"  ✓ queries.csv saved — {len(queries):,} rows")

# ── 3. Clean SemEval posts ──
print("Cleaning SemEval posts...")
posts_df['text_clean'] = posts_df['text_clean'].apply(light_clean)
posts_df = posts_df[posts_df['text_clean'].notna()]
posts_df = posts_df[posts_df['text_clean'].str.len() > 3].reset_index(drop=True)
posts_df.to_csv(DRIVE + 'semeval/train_dev_sets/posts_clean.csv', index=False)
print(f"  ✓ posts_clean.csv saved — {len(posts_df):,} rows")

# ── Summary ──
print("\n" + "="*45)
print("PREPROCESSING COMPLETE — never run again")
print("="*45)
print(f"  unified_corpus.csv  → {len(unified):,} rows")
print(f"  queries.csv         → {len(queries):,} rows")
print(f"  posts_clean.csv     → {len(posts_df):,} rows")
print(f"  qrels.csv           → untouched (IDs only)")
print(f"  pairs.csv           → untouched (IDs only)")

Cleaning unified corpus...
  ✓ unified_corpus.csv saved — 393,424 rows
Cleaning DisastIR queries...
  ✓ queries.csv saved — 9,600 rows
Cleaning SemEval posts...
  ✓ posts_clean.csv saved — 21,853 rows

PREPROCESSING COMPLETE — never run again
  unified_corpus.csv  → 393,424 rows
  queries.csv         → 9,600 rows
  posts_clean.csv     → 21,853 rows
  qrels.csv           → untouched (IDs only)
  pairs.csv           → untouched (IDs only)
